# Task 4 – Voiceprint Verification Model (Training)

**Owner:** Santhiana
**Reads:** `data/audio/processed/audio_features.csv` (made by `src/audio_pipeline.py`)
**Writes:** `models/voice_model.joblib`

**Goal:** given a recording, say **which team member** it belongs to, *or* say **"unknown"** if it doesn't clearly match anyone. The second part is what makes the **unauthorized attempt** fail.

**Scores we track** (same as the product model, so all three models are compared the same way):
- **Accuracy:** the share of correct guesses. With 4 members, guessing at random gives about 25%.
- **F1-score (macro):** accuracy that treats every member fairly.
- **Loss (log loss):** how confident-and-wrong the model is. Lower is better, and random guessing gives about 1.39 (ln 4).

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))   # reuse the project code in src/
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import config as C

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
import joblib
from sklearn.metrics import ConfusionMatrixDisplay
from identity_model import train_identity_model, save_bundle, identify, feature_columns
from audio_pipeline import audio_to_features

# audio_pipeline.py names the columns speaker / utterance / variant / filename;
# rename them to the shared names used by identity_model.py (same as the face model)
df = (pd.read_csv(C.AUDIO_FEATURES_CSV)
        .rename(columns={"speaker": "member", "utterance": "source_file", "variant": "augmentation"})
        .drop(columns=["filename"]))
print(df.shape)
df.iloc[:, :8].head()

In [ ]:
print("Rows per member:"); print(df.member.value_counts().to_string())
print("\nOriginal recordings per member:"); print(df.groupby("member")["source_file"].nunique().to_string())
print("\nFeatures:", len(feature_columns(df)), "| missing values:", df.isna().sum().sum())

## 1. A fair test: split by original recording

Every recording appears 6 times in the table (the clean original plus its 5 augmented copies).
- **If we split rows at random**, a flipped copy could land in training while the original lands in testing. The model would effectively have seen the answer, and the score would look much better than it really is.
- **So we split by `source_file`** using `StratifiedGroupKFold`: every copy of one recording always stays on the same side, and each fold still contains every member.

This is the same idea the product model uses when it splits by customer.

The number of folds equals the smallest number of original recordings any member has. Each fold therefore tests one unseen recording per member.

## 2. Compare two models and pick the best

| Model | Why we try it |
|---|---|
| **Logistic Regression** (with standard scaling) | Simple, fast, and gives well-calibrated probabilities, which matters for the confidence threshold. |
| **Random Forest** (300 trees, at least 2 rows per leaf) | Handles non-linear patterns and needs no scaling. The leaf limit stops it from memorising single rows. |

The model with the best cross-validated **F1** wins. If the F1 scores tie, the lower **loss** wins.

In [ ]:
bundle = train_identity_model(df, group_col="source_file")

In [ ]:
b = bundle
summary = pd.DataFrame([{"set": "train (refit on all)", **b["train_scores"]},
                        {"set": "cross-validation (unseen files)", **b["cv_scores"]}]).set_index("set")
ax = summary.plot(kind="bar", figsize=(8, 4), rot=0)
ax.axhline(0.25, color="grey", ls="--", lw=1, label="random accuracy (0.25)")
ax.set_title(f"{b['model_name']}: train vs cross-validation"); ax.legend(loc="upper right"); plt.show()
summary.round(3)

**How to read it:**
- The **train** scores come from the model refit on every row, so they are optimistic.
- The **cross-validation** scores come from recordings the model never saw, and these are the honest numbers.
- A big gap between the two (for example 1.00 train vs 0.70 CV) means the model memorises. A small gap means it generalises.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ConfusionMatrixDisplay(b["confusion"], display_labels=sorted(df.member.unique())).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Who was predicted (cross-validation)")

conf = b["oof_proba"].max(1)
pred = np.array(sorted(df.member.unique()))[b["oof_proba"].argmax(1)]
right = pred == b["oof_true"]
axes[1].hist(conf[right], bins=15, alpha=0.7, label="correct guesses", color="green")
axes[1].hist(conf[~right], bins=15, alpha=0.7, label="wrong guesses", color="red")
axes[1].axvline(b["threshold"], color="black", ls="--", label=f"threshold = {b['threshold']:.2f}")
axes[1].set_xlabel("model confidence (top probability)"); axes[1].set_ylabel("count")
axes[1].set_title("Confidence of correct vs wrong guesses"); axes[1].legend()
plt.tight_layout(); plt.show()

## 3. Decision rule: when do we say "unknown"?

A normal classifier **always** picks one of the 4 members, even for a stranger. To be able to say *no*, the system uses two checks:

1. **Confidence threshold:** the model's top probability must be at least the threshold.
   - The threshold is set automatically to the confidence reached by **90% of correct** cross-validation guesses. In other words, we accept about 90% of genuine attempts.
   - It is kept between 0.40 and 0.85.
   - In the right-hand histogram, the wrong guesses (red) should mostly fall to the left of the dashed line, which means they get rejected.
2. **Novelty check (Local Outlier Factor):** this asks whether the input looks like *anything* in the training data. A stranger's recording sits far from every member's points, so it's flagged even if the classifier happens to be confident.

Access is granted only when both checks pass.

In [ ]:
print(f"Genuine attempts accepted (cross-validation): {b['genuine_accept_rate']:.0%}")
print(f"Accepted as the WRONG member               : {b['wrong_member_accept_rate']:.0%}")
print(f"Flagged as 'unknown' by the novelty check   : {b['oof_novel'].mean():.0%} of genuine attempts")

## 4. Unauthorized attempt: a non-member must be rejected

In [ ]:
files = [p for p in sorted(C.AUDIO_UNAUTH_DIR.glob("*")) if p.suffix.lower() in C.AUDIO_EXTS]
if not files:
    print("No file in", C.AUDIO_UNAUTH_DIR, "- add a non-member sample (or run: python src/make_unauthorized_samples.py)")
for p in files:
    r = identify(b, audio_to_features(p))
    print(f"{p.name}: {'ACCEPTED as ' + r['member'] if r['accepted'] else 'REJECTED'}  ->  {r['reason']}")
    print("   probabilities:", r["probabilities"])

## 5. Save the model

In [ ]:
save_bundle(b, C.VOICE_MODEL_PATH)
print("Saved:", C.VOICE_MODEL_PATH, "| keys:", list(joblib.load(C.VOICE_MODEL_PATH).keys()))

## Summary
- **Validation:** grouped by original recording, so augmented copies never leak into the test fold.
- **Model choice:** Logistic Regression and Random Forest were compared, and the best by cross-validated F1 was kept.
- **Metrics reported:** accuracy, F1, loss (train vs CV) and a confusion matrix.
- **Rejecting strangers:** the confidence threshold plus the novelty check let the model answer *"unknown"*, which the command-line app uses to deny access.
- **Saved to:** `models/voice_model.joblib`, which `app/cli.py` loads.

The same steps can be run without Jupyter: `python src/train_voice_model.py --rebuild`.